# Configuration

## Install necessary libraries from python

In [1]:
import os
import pickle
import subprocess
import shutil

In [2]:
%%capture
!pip install transformers datasets torch
!pip install google-generativeai python-dotenv -q
!pip install git+https://github.com/huggingface/accelerate
# !pip install dgl==2.0.0 -f https://data.dgl.ai/wheels/cu121/repo.html
!sudo apt-get -q install graphviz graphviz-dev
!pip install -q pygraphviz
!pip install -q slither-analyzer==0.8.0
!pip install dgl==1.1.2
!pip install py-solc==3.2.0
!pip install networkx==2.5.1
!solc-select install 0.4.25
!solc-select use 0.4.25
!which solc && solc --version

In [3]:
# %%capture
file_path = '/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/sc_versions.pkl'
with open(file_path, 'rb') as f:
    sc_versions = pickle.load(f)

destination_path = '/content/ge-sc/artifacts'
for sc_version in sc_versions:

    print(sc_version)
    try:
        subprocess.run(['solc-select', 'install', sc_version])
        solc_compiler = os.path.expanduser(f'~/.solc-select/artifacts/solc-{sc_version}')
        shutil.copytree(solc_compiler, destination_path, dirs_exist_ok=True)
    except Exception as e:
        print(sc_version)
        print(e)

0.4.20
0.4.99
0.4.99
[Errno 2] No such file or directory: '/root/.solc-select/artifacts/solc-0.4.99'
0.4.9
0.4.22
0.4.19
0.4.2
0.4.11
0.4.10
0.4.23
0.5.5
0.5.2
0.5.7
0.5.0
0.4.8
0.4.7
0.5.8
0.4.4
0.4.15
0.4.24
0.4.12
0.4.16
0.4.13
0.4.26
0.5.9
0.5.4
0.5.6
0.4.0
0.5.3
0.4.21
0.5.1
0.4.17
0.8.0
0.4.18
0.4.25
0.4.14
0.4.6


## Import Python libraries

In [4]:
from concurrent.futures import ThreadPoolExecutor
from random import sample
import json
import multiprocessing
import traceback
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import dgl
import shutil
import random
import re
import logging
from copy import deepcopy
from os.path import join
from scipy.integrate._ivp.radau import C
from slither.slither import Slither
from collections import defaultdict
from networkx.algorithms import cluster
from slither.core.cfg.node import Node, NodeType
from slither.printers.call import call_graph
from slither.printers.abstract_printer import AbstractPrinter
from slither.core.declarations.solidity_variables import SolidityFunction
from slither.core.declarations.function import Function
from slither.core.variables.variable import Variable
import glob
from multiprocessing import Pool as ThreadPool
from functools import partial
import torch.nn as nn
import torch
from torch import Tensor
import torch.nn.functional as F
from dgl.nn import GraphConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import pickle
import random
import numpy as np
from tqdm import tqdm
from shutil import copy
from re import L
from typing import Pattern

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


# Solidity code to Graph

## Generate call-graph

In [5]:
logger = logging.getLogger("Slither-simil")

def seed_everything(seed: int):
    # Hàm đặt seed cố định cho tất cả các thư viện để kết quả có thể tái tạo lại được
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)  # Cố định seed cho thư viện random
    # os.environ['PYTHONHASHSEED'] = str(seed)  # Cố định hash seed của Python
    np.random.seed(seed)  # Cố định seed cho NumPy
    torch.manual_seed(seed)  # Cố định seed cho PyTorch CPU
    torch.cuda.manual_seed(seed)  # Cố định seed cho PyTorch GPU
    torch.backends.cudnn.deterministic = True  # Đảm bảo tính nhất quán của thuật toán cuDNN
    torch.backends.cudnn.benchmark = True  # Tối ưu hóa tốc độ tính toán

# get sol version
def get_solc_version(source):
    pattern =  re.compile(r'\d.\d.\d+')
    with open(source, 'r') as f:
        line = f.readline()
        while line:
            if 'pragma solidity' in line:
                if len(pattern.findall(line)) > 0:
                    return pattern.findall(line)[0]
                else:
                    return '0.4.25'
            line = f.readline()
    return '0.4.25'

# Contract function node
def _function_node(contract, function, filename_input):
    node_function_source_code_start = function.source_mapping['start']
    node_function_source_code_length = function.source_mapping['length']
    node_info = {
        'node_id': f"{filename_input}_{contract.id}_{contract.name}_{function.full_name}",
        'label': f"{filename_input}_{contract.name}_{function.full_name}",
        'function_fullname': function.full_name,
        'contract_name': contract.name,
        'source_file': filename_input,
        'node_source_code_start': node_function_source_code_start,
        'node_source_code_length': node_function_source_code_length,
        'visibility': function.visibility
    }
    return node_info

# Solidity function node
def _solidity_function_node(solidity_function):
    node_info = {
        'node_id': f"[Solidity]_{solidity_function.full_name}",
        'label': f"[Solidity]_{solidity_function.full_name}",
        'function_fullname': solidity_function.full_name,
        'contract_name': None,
        'source_file': None,
        'node_source_code_start': None,
        'node_source_code_length': None,
        'visibility': 'public'
    }
    return node_info

# return node info from a node tupple
def _get_node_info(tuple_node):
    if tuple_node[0][0] == 'node_id':
        node_id = tuple_node[0][1]
    if tuple_node[1][0] == 'label':
        node_label = tuple_node[1][1]
    if tuple_node[2][0] == 'function_fullname':
        function_fullname = tuple_node[2][1]
    if tuple_node[3][0] == 'contract_name':
        contract_name = tuple_node[3][1]
    if tuple_node[4][0] == 'source_file':
        source_file = tuple_node[4][1]
    if tuple_node[5][0] == 'node_source_code_start':
        node_function_source_code_start = tuple_node[5][1]
    if tuple_node[6][0] == 'node_source_code_length':
        node_function_source_code_length = tuple_node[6][1]
    if tuple_node[7][0] == 'visibility':
        visibility = tuple_node[7][1]

    if 'fallback' in node_id:
        node_type = 'fallback_function'
    elif '[Solidity]' in node_id:
        node_type = 'fallback_function'
    else:
        node_type = 'contract_function'

    return node_id, node_label, node_type, function_fullname, contract_name, source_file, node_function_source_code_start, node_function_source_code_length, visibility

# return edge info from a contract call tuple
def _add_edge_info_to_nxgraph(contract_call, nx_graph):
    source = contract_call[0]
    source_node_id, source_label, source_type, source_function_fullname, source_contract_name, \
    source_source_file, source_node_function_source_code_start, source_node_function_source_code_length, source_visibility = _get_node_info(source)

    if source_node_id not in nx_graph.nodes():
        nx_graph.add_node(source_node_id, label=source_label, node_type=source_type,
                          node_source_code_start=source_node_function_source_code_start, node_source_code_length=source_node_function_source_code_length,
                          function_fullname=source_function_fullname,
                          function_vis=source_visibility, contract_name=source_contract_name,
                          source_file=source_source_file)

    target = contract_call[1]
    target_node_id, target_label, target_type, target_function_fullname, target_contract_name, \
    target_source_file, target_node_function_source_code_start, target_node_function_source_code_length,  target_visibility = _get_node_info(target)

    if target_node_id not in nx_graph.nodes():
        nx_graph.add_node(target_node_id, label=target_label, node_type=target_type,
                          node_source_code_start=target_node_function_source_code_start, node_source_code_length=target_node_function_source_code_length,
                          function_fullname=target_function_fullname,
                          function_vis=target_visibility, contract_name=target_contract_name,
                          source_file=target_source_file)

    edge_type = contract_call[2]
    edge_label = contract_call[3]

    nx_graph.add_edge(source_node_id, target_node_id, label=edge_label, edge_type=edge_type)

def _process_internal_call(
    contract,
    function,
    internal_call,
    contract_calls,
    solidity_functions,
    solidity_calls,
    filename_input
):
    if isinstance(internal_call, (Function)):
        contract_calls[contract].add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_function_node(contract, internal_call, filename_input).items()),
                'internal_call',
                'internal_call'
            )
        )

    elif isinstance(internal_call, (SolidityFunction)):
        solidity_functions.add(tuple(_solidity_function_node(internal_call).items()))
        solidity_calls.add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_solidity_function_node(internal_call).items()),
                'solidity_call',
                'solidity_call'
            )
        )

def _process_external_call(
    contract,
    function,
    external_call,
    contract_functions,
    external_calls,
    all_contracts,
    filename_input
):
    external_contract, external_function = external_call
    if not external_contract in all_contracts:
        return

    if isinstance(external_function, (Variable)):
        contract_functions[external_contract].add(tuple(
                _function_node(external_contract, external_function, filename_input).items()))

    external_calls.add(
        (
            tuple(_function_node(contract, function, filename_input).items()),
            tuple(_function_node(external_contract, external_function, filename_input).items()),
            'external_call',
            'external_call'
        )
    )

def _process_function(
    contract,
    function,
    contract_functions,
    contract_calls,
    solidity_functions,
    solidity_calls,
    external_calls,
    all_contracts,
    filename_input
):
    contract_functions[contract].add(tuple(
        _function_node(contract, function, filename_input).items())
    )
    for internal_call in function.internal_calls:
        _process_internal_call(
            contract,
            function,
            internal_call,
            contract_calls,
            solidity_functions,
            solidity_calls,
            filename_input
        )
    for external_call in function.high_level_calls:

        _process_external_call(
            contract,
            function,
            external_call,
            contract_functions,
            external_calls,
            all_contracts,
            filename_input
        )

def _process_functions(functions, filename_input, vulnerabilities_in_sc=None):
    contract_functions = defaultdict(set)  # contract -> contract functions nodes
    contract_calls = defaultdict(set)  # contract -> contract calls edges

    solidity_functions = set()  # solidity function nodes
    solidity_calls = set()  # solidity calls edges

    external_calls = set()  # external calls edges

    all_contracts = set()
    for function in functions:
        all_contracts.add(function.contract_declarer)

    for function in functions:
        _process_function(
            function.contract_declarer,
            function,
            contract_functions,
            contract_calls,
            solidity_functions,
            solidity_calls,
            external_calls,
            all_contracts,
            filename_input
        )

    all_contracts_graph = nx.MultiDiGraph()
    for contract in all_contracts:
        if len(contract_functions[contract]) > 0:
            for contract_function in contract_functions[contract]:
                node_id, node_label, node_type, function_fullname, contract_name, source_file, \
                node_function_source_code_start, node_function_source_code_length, source_visibility = _get_node_info(contract_function)

                all_contracts_graph.add_node(node_id, label=node_label, node_type=node_type,
                                  node_source_code_start=node_function_source_code_start, node_source_code_length=node_function_source_code_length,
                                  function_fullname=function_fullname, function_vis=source_visibility, contract_name=contract_name,
                                  source_file=source_file)

        if len(contract_calls[contract]) > 0:
            for contract_call in contract_calls[contract]:
                _add_edge_info_to_nxgraph(contract_call, all_contracts_graph)

    if len(external_calls) > 0:
        for external_call in external_calls:
            _add_edge_info_to_nxgraph(external_call, all_contracts_graph)

    return all_contracts_graph

## Generate control-flow graph

In [6]:
def get_node_info(node):
    node_label = "Node Type: {}\n".format(str(node.type))
    node_type = str(node.type)
    if node.expression:
        node_label += "\nEXPRESSION:\n{}\n".format(node.expression)
        node_expression = str(node.expression)
    else:
        node_expression = None
    if node.irs:
        node_label += "\nIRs:\n" + "\n".join([str(ir) for ir in node.irs])
        node_irs = "\n".join([str(ir) for ir in node.irs])
    else:
        node_irs = None

    # print(node_label)
    node_source_code_start = node.source_mapping['start']
    node_source_code_length = node.source_mapping['length']

    return node_label, node_type, node_expression, node_irs, node_source_code_start, node_source_code_length

## Merging Cfgs to Fcgs

In [7]:
def mapping_cfg_and_cg_node_labels(cfg, call_graph):
    dict_node_label_cfg_and_cg = {}

    for node, node_data in cfg.nodes(data=True):
        if node_data['node_type'] == 'FUNCTION_NAME':
            if node_data['label'] not in dict_node_label_cfg_and_cg:
                dict_node_label_cfg_and_cg[node_data['label']] = None

            dict_node_label_cfg_and_cg[node_data['label']] = {
                'cfg_node_id': node,
                'cfg_node_type': node_data['node_type']
            }

    for node, node_data in call_graph.nodes(data=True):
        if node_data['label'] in dict_node_label_cfg_and_cg:
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_id'] = node
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_type'] = node_data['node_type'].upper()
        else:
            print(node_data['label'], ' is not existing.')

    temp_dict = dict(dict_node_label_cfg_and_cg)
    for key, value in temp_dict.items():
        if 'call_graph_node_id' not in value or 'call_graph_node_type' not in value:
            dict_node_label_cfg_and_cg.pop(key, None)

    return dict_node_label_cfg_and_cg

def add_new_cfg_edges_from_call_graph(cfg, dict_node_label, call_graph):
    list_new_edges_cfg = []
    for source, target, edge_data in call_graph.edges(data=True):
        source_cfg = None
        target_cfg = None
        edge_data_cfg = edge_data
        for value in dict_node_label.values():
            if value['call_graph_node_id'] == source:
                source_cfg = value['cfg_node_id']

            if value['call_graph_node_id'] == target:
                target_cfg = value['cfg_node_id']

        if source_cfg is not None and target_cfg is not None:
            list_new_edges_cfg.append((source_cfg, target_cfg, edge_data_cfg))

    cfg.add_edges_from(list_new_edges_cfg)

    return cfg

def update_cfg_node_types_by_call_graph_node_types(cfg, dict_node_label):
    for value in dict_node_label.values():
        cfg_node_id = value['cfg_node_id']
        cfg.nodes[cfg_node_id]['node_type'] = value['call_graph_node_type']

# Build Agent

In [8]:
import os
import re
import json
import time
import logging
import google.generativeai as genai
from dotenv import load_dotenv
from typing import List, Dict, Union, Tuple

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
except ImportError:
    print("Warning: 'kaggle_secrets' library not found. Assuming not running in Colab.")
    user_secrets = None

# Load API keys from Colab Secrets or environment variables
API_SECRET_NAMES = ["Gemini_Kien", "Gemini_Manh_Vai_Cut", "Gemini_Quang_9a3", "Gemini_Quang_71103", "Gemini_Kien_Junior", "Gemini_Minh_Junior"]

# Class to manage API keys and rotate between them
class APIKeyManager:
    def __init__(self, api_keys: List[str]):
        self.api_keys = [key for key in api_keys if isinstance(key, str) and key.strip()]
        self.current_index = 0
        self.retry_count = 0
        self.max_retries = len(self.api_keys) * 2 if self.api_keys else 1
        self.key_status = {
            i: {"is_limited": False, "last_error": None, "error_time": None}
            for i in range(len(self.api_keys))
        }

    def get_current_key(self) -> str:
        """Return the current API key."""
        return self.api_keys[self.current_index] if self.api_keys else ""

    def rotate_key(self) -> str:
        """Rotate to the next key and return it."""
        if not self.api_keys:
            return ""
        self.key_status[self.current_index]["is_limited"] = True
        self.key_status[self.current_index]["error_time"] = time.time()
        self.current_index = (self.current_index + 1) % len(self.api_keys)
        self.retry_count += 1
        return self.get_current_key()

    def has_valid_keys(self) -> bool:
        """Check if any valid keys are available."""
        return len(self.api_keys) > 0 and self.retry_count < self.max_retries

    def mark_current_key_error(self, error_msg: str) -> None:
        """Mark the current key as having an error."""
        if self.api_keys:
            self.key_status[self.current_index]["last_error"] = error_msg
            self.key_status[self.current_index]["error_time"] = time.time()

    def get_key_status(self) -> Dict:
        """Return the status of all keys."""
        return {
            f"key_{i+1}": {
                "is_limited": status["is_limited"],
                "last_error": status["last_error"],
                "error_time": time.strftime('%H:%M:%S', time.localtime(status["error_time"]))
                if status["error_time"]
                else None
            }
            for i, status in self.key_status.items()
        }

# Initialize API keys
api_keys = []
if user_secrets:
    for secret_name in API_SECRET_NAMES:
        try:
            key = user_secrets.get_secret(secret_name)
            if key:
                api_keys.append(key)
        except Exception as e:
            print(f"Warning: Could not access secret '{secret_name}': {e}")
else:
    import os
    for secret_name in API_SECRET_NAMES:
        key = os.getenv(secret_name)
        if key:
            api_keys.append(key)
        else:
            print(f"Warning: Environment variable '{secret_name}' not found.")

# Initialize API Key Manager
api_manager = APIKeyManager(api_keys)

if not api_manager.has_valid_keys():
    print("Error: No valid API keys found.")
    print(f"Please add at least one valid API key with names: {', '.join(API_SECRET_NAMES)}")

    def classify_functions_vulnerability(functions_dict: Dict[str, str]) -> Dict[str, List[str]]:
        print("No valid API keys available. Cannot classify.")
        return {
            "Vulnerable": [],
            "NonVulnerable": []
        }
else:
    import google.generativeai as genai
    from google.generativeai import types
    genai.configure(api_key=api_manager.get_current_key())

    def classify_functions_vulnerability(vul_type: str, functions_dict: Dict[str, str]) -> Tuple[Dict[str, int], Dict[str, str]]:
        """
        Phân tích danh sách các hàm Solidity từ một file duy nhất bằng API Gemini để phân loại hàm dễ bị tổn thương hoặc không.
        Xử lý từng hàm riêng lẻ và lưu nhãn cùng lý do vào file .fcg tương ứng.
    
        Args:
            vul_type: Loại lỗ hổng ("timestamp" hoặc "reentrancy").
            functions_dict: Danh sách các đoạn mã hàm cần phân tích.
    
        Returns:
            Tuple[Dict[str, int], Dict[str, str]]: Một tuple chứa:
                - Danh sách nhãn (1: dễ bị tổn thương, 0: không dễ bị tổn thương).
                - Danh sách chuỗi lý do cho từng hàm.
    
        Raises:
            APIError: Nếu phân loại qua API thất bại sau tất cả các lần thử và khóa API.
        """
        if not api_manager.has_valid_keys():
            error_msg = "Không có khóa API hợp lệ để phân loại."
            logging.error(error_msg)
            raise APIError(error_msg)

        labels = {}
        reasons = {}
        for idx, (func_name, func_code) in enumerate(functions_dict.items()):
            logging.info(f"Phân loại hàm {idx + 1}/{len(functions_dict)} cho {vul_type} trong file")
            print(f"Phân loại hàm {idx + 1}/{len(functions_dict)} cho {vul_type} trong file")
            
            # Định nghĩa prompt dựa trên loại lỗ hổng
            if vul_type.lower() == "timestamp":
                prompt = f"""You are a smart contract security auditor. Your task is to analyze a single provided Solidity function, which contains `block.timestamp` or `now`, to determine if it has a **timestamp dependence vulnerability**. Follow the steps below to perform explicit reasoning and provide the reasoning in the output before the final answer.
    
    ---
    
    ### Step 1: Understand the Labeling Criteria
    
    The function is vulnerable to timestamp dependence if it satisfies either:
    - **TimestampAssign**: `block.timestamp` or `now` is assigned to a variable, and that variable is used in subsequent operations (e.g., return statements, calculations, or storage updates).
    - **OR**
    - **TimestampContaminate**: `block.timestamp` or `now` influences the return value or triggers a financial operation (e.g., `transfer`, `call.value`).
    
    **Key Rules:**
    - If `block.timestamp` or `now` is only used to throw an error (e.g., in `require`, `assert`, or `revert`), label = 0 (not vulnerable).
    - Otherwise, if either TimestampAssign or TimestampContaminate is met, label = 1 (vulnerable).
    
    ---
    
    ### Step 2: Analyze the Function Line-by-Line with Explicit Reasoning
    
    Analyze the provided function line-by-line and document your reasoning as follows:
    
    1. **Check for Assignment (TimestampAssign):**
       - Identify if `block.timestamp` or `now` is assigned to a variable.
       - If **yes**, check if that variable is used later (e.g., in a return statement, calculation, or storage update).
         - If **used**, note: "Vulnerable (TimestampAssign: `block.timestamp` or `now` assigned to variable and used in subsequent operation). Label = 1."
         - If **not used**, note: "No usage of assigned variable."
       - If **no assignment**, note: "No assignment of `block.timestamp` or `now`."
    
    2. **Check for Contamination (TimestampContaminate):**
       - Check if `block.timestamp` or `now` influences the return value (e.g., used in a conditional that determines what is returned).
         - If **yes**, note: "Vulnerable (TimestampContaminate: influences return value). Label = 1."
       - Check if it triggers a financial operation (e.g., `transfer`, `call.value`) based on its value.
         - If **yes**, note: "Vulnerable (TimestampContaminate: triggers financial operation). Label = 1."
       - Check if it is only used to throw an error (e.g., `require(block.timestamp > X)`).
         - If **yes**, note: "Not vulnerable (only throws error). Label = 0."
       - If none of the above apply, note: "Not vulnerable (no assignment or contamination). Label = 0."
    
    3. **Assign the Final Label:**
       - Based on your reasoning, assign a single label (0 or 1).
    
    ---
    
    ### Step 3: Reference Examples
    
    Use these examples to guide your reasoning:
    
    - **Vulnerable (label = 1, TimestampAssign)**:
      ```
      function closeRound() public returns (uint256) {{
          uint256 closingTime = block.timestamp + 1;
          return closingTime;
      }}
      ```
      *Reasoning: `block.timestamp` is assigned to `closingTime`, which is used in the return statement. Vulnerable (TimestampAssign). Label = 1.*
    
    - **Not Vulnerable (label = 0, only throws error)**:
      ```
      function withdrawal() public {{
          require(block.timestamp >= lock);
      }}
      ```
      *Reasoning: `block.timestamp` is only used in a `require` statement to throw an error. Not vulnerable. Label = 0.*
    
    - **Vulnerable (label = 1, TimestampContaminate)**:
      ```
      function getState() public returns (uint) {{
          if (block.timestamp < startsAt) return 1;
          return 2;
      }}
      ```
      *Reasoning: `block.timestamp` determines the return value via a conditional. Vulnerable (TimestampContaminate). Label = 1.*
    
    - **Vulnerable (label = 1, TimestampContaminate, financial operation)**:
      ```
      function transfer(address _to, uint _value) public {{
          if (now > releaseDate) {{
              _to.transfer(_value);
          }}
      }}
      ```
      *Reasoning: `now` triggers a financial operation (`transfer`). Vulnerable (TimestampContaminate). Label = 1.*
    
    ---
    
    ### Step 4: Analyze the Provided Function
    
    You will be given a single Solidity function containing `block.timestamp` or `now`. Analyze it line-by-line using the reasoning steps above.
    
    ---
    
    ### Step 5: Format Your Response
    
    - First, provide the explicit reasoning for the function, detailing each step of your analysis as described in Step 2.
    - Then, provide the final answer as a single number (0 or 1) enclosed within `<RESULT_START>` and `<RESULT_END>` markers.
    - Do not include any additional text or comments outside the reasoning and the result markers.
    
    **Example Output:**
    ```
    Reasoning: `block.timestamp` is assigned to `closingTime`, which is used in the return statement. Vulnerable (TimestampAssign: `block.timestamp` assigned to variable and used in subsequent operation). Label = 1.
    <RESULT_START>{{"label": 1, "reason": "`block.timestamp` is assigned to `closingTime`, which is used in the return statement. Vulnerable (TimestampAssign: `block.timestamp` assigned to variable and used in subsequent operation)."}}<RESULT_END>
    
    ```
    
    ---
    
    ### Solidity Function to Analyze
    
    ```solidity
    {func_code}
    ```
    """
            else:  # Reentrancy
                prompt1 = """You are a highly advanced Smart Contract Security Auditor. Your mission is to meticulously analyze a single provided Solidity function or modifier. This function/modifier contains a `call.value` (or its equivalent, e.g., `call{{value: ...}}`).

Your primary goal is to identify **any potential execution path** where this `call.value` could be misused to enable a reentrancy attack. This means you need to think like an attacker: if the `call.value` transfers control (and potentially Ether) to an external, potentially malicious contract, could that contract call back into the original function (or another function sharing critical state with it) *before* the original function has fully and safely completed all its intended operations, including updating its internal state **and emitting all relevant events or other external notifications**?

---

### Step 1: Understand the Core Objective – Finding Re-entrant Paths Leading to Incomplete Operations

A reentrancy vulnerability exists if:
1.  The function makes an external call using `call.value` to an address that could be controlled by an attacker.
2.  **Before** this external call, the function has **not yet fully completed all its critical effects**. This includes:
    *   Updating all relevant internal state variables (e.g., debiting a balance, setting a lock, incrementing a usage counter).
    *   **AND, crucially, emitting all necessary events or performing other external notifications** that signify the successful completion or outcome of the current logical operation.
3.  The attacker's contract, upon receiving the call, can call back into the vulnerable function (or another related function) and find the contract in a state where the original operation is effectively "in-flight" or incomplete from an external observer's perspective (e.g., internal state changed but the corresponding event not yet emitted).
4.  This re-entry into an "in-flight" operation leads to an exploitable condition. This could be direct fund siphoning, state corruption, or causing **misleading sequences of events/operations** that break the logic of reliant contracts or off-chain systems.
5.  Standard access controls (like `onlyOwner` on the *entire function*) or `private` visibility might mitigate this, but your analysis should focus on the interaction flow itself assuming the function is callable by a potential attacker.

---

### Step 2: Deep Dive Analysis – Tracing Potential Re-entrant Flows and Incomplete Operations

For the provided Solidity code:

1.  **Identify the External Call:**
    *   Pinpoint the `call.value` invocation.
    *   Note the recipient and the value. Is the recipient arbitrary? Is the value potentially non-zero?

2.  **Analyze All Effects and Their Order Relative to the External Call:**
    *   What checks are performed *before* the `call.value`?
    *   What internal state variables are read *before* the `call.value`?
    *   **Crucially, itemize all intended effects of this function/operation:**
        *   What internal state variables are *written to or updated* (e.g., balance changes)?
        *   What *events are intended to be emitted* to signal the operation's outcome?
        *   Are there any other external notifications or calls planned as part of this operation?
    *   **Order of Operations Check:**
        *   Are all internal state updates performed **before** the `call.value`?
        *   Are all critical events (that signify the completion/outcome of *this specific operation*) emitted **before** the `call.value`?
        *   If not, note which effects (internal state updates or event emissions) occur *after* the `call.value`.

3.  **Hypothesize Re-entry During "In-Flight" Operation:**
    *   Assume the recipient of the `call.value` is a malicious contract.
    *   Imagine this malicious contract *immediately* calls back into the function currently being analyzed (or another public/external function in the same contract, or triggers interaction from another contract).
    *   At this point of re-entry, consider that the original operation has had some effects (e.g., internal balance updates) but *not all* (e.g., the final `Transfer` event hasn't been emitted yet).

4.  **Assess Exploitability upon Re-entry (Considering Incomplete Operations & Event Ordering):**
    *   If the function/contract is re-entered while the original operation is "in-flight":
        *   Could the attacker exploit internal state that is updated but not yet publicly "committed" via an event?
        *   Could the attacker cause their own operations/events to be recorded *before* the event corresponding to the original, outer operation is emitted? This creates an **out-of-order event sequence**.
        *   How would such an out-of-order event sequence affect reliant systems (e.g., exchanges, UIs, other contracts monitoring events)? Could it lead to misinterpretation of state, incorrect accounting, or bypass of logic that depends on event order?
        *   Are there any reentrancy guards (e.g., a mutex/lock) that robustly prevent re-entry *before all effects, including final event emissions*, are completed? A lock set before internal state changes but released before event emission is not sufficient if the event order itself is critical.

5.  **Consider Access Control Context:**
    *   Does the function have strict access controls or `private` visibility that would inherently prevent an untrusted party from initiating this sequence? If so, state this, but still analyze the pattern.

6.  **Synthesize Findings for Vulnerability:**
    *   Based on the above, is there a plausible path where re-entry occurs *before the original operation has completed all its effects (including emitting its final, defining events)*, leading to an exploitable condition (e.g., misleading event order, state inconsistency from an observer's perspective)?
    *   If `call.value(0)` is used: while it doesn't transfer Ether to drain directly, the primary concern here is the re-entry *before completion of all effects of the outer function call*.

---

### Step 3: Provide Detailed Reasoning and Label

1.  **Explicit Reasoning:** Document your thought process step-by-step, covering the points from Step 2. Clearly explain:
    *   The point of external interaction (`call.value`).
    *   The sequence of internal state changes, the external call, and subsequent event emissions or other effects.
    *   How a re-entrant call could occur *before the original operation is fully complete (including its public notification via events)*.
    *   How this re-entry could lead to an exploitable condition, explicitly mentioning issues like out-of-order event logging or interaction with an incompletely updated state from an external viewpoint.
    *   The role (or lack thereof) of any reentrancy guards or relevant access controls.

2.  **Final Label:**
    *   If you identify a plausible re-entrant path leading to an exploitable condition because the `call.value` occurs *before all critical effects of the operation (including internal state updates AND final event emissions) are completed*, assign **Label = 1** (Vulnerable).
    *   If all critical internal state updates AND all critical event emissions signifying the operation's outcome occur *before* the `call.value` (full Checks-Effects-Interactions pattern including events), OR if there's a robust reentrancy guard preventing re-entry before full completion, OR if strict access control/`private` visibility prevents untrusted interaction, assign **Label = 0** (Not Vulnerable under these conditions).

---

### Step 4: Format Your Response

-   First, provide the explicit reasoning for the function or modifier, detailing your analysis as described in Step 2 and 3.1.
-   Then, provide the final answer as a JSON object containing `label` (an integer, 0 or 1) and `reason` (a string, which is a concise summary of why the label was chosen, highlighting the core finding about re-entrant paths and incomplete operations) enclosed within `<RESULT_START>` and `<RESULT_END>` markers.
-   Do not include any additional text or comments outside the reasoning and the result markers.

**Example Output (Illustrative for a vulnerable function like the one discussed):**
```
Reasoning:
1.  **External Call:** The function uses `_to.call.value(0)(...)` for an external interaction. `_to` can be an attacker's contract.
2.  **Order of Effects:** Internal balances are updated *before* this external call. However, the critical `Transfer` event, which signals the completion of the token transfer to external observers, is emitted *after* the external call.
3.  **Hypothesized Re-entry:** An attacker controlling `_to` can execute code upon receiving the call. This code can call back into the original contract (or trigger interactions with it) *before* the `Transfer` event for the outer call is emitted.
4.  **Exploitability:** If re-entry occurs, any events emitted or state changes made during the re-entrant call will be logged on the blockchain *before* the `Transfer` event of the original, outer call. This creates an out-of-order event sequence. Systems relying on the correct chronological order of `Transfer` events (e.g., exchanges, wallets, other smart contracts) can be misled, leading to incorrect accounting, broken logic, or other exploits based on this inconsistent view of operations. The internal balances are consistent, but the public record of the operation (the event) is manipulable in its timing relative to other actions. No reentrancy guard prevents re-entry before the event emission.
5.  **Access Control:** The function is public, allowing this interaction.
6.  **Synthesis:** The function is vulnerable because the external call precedes the emission of the `Transfer` event, allowing an attacker to re-enter and cause events to be logged in a misleading order.

This function is vulnerable to reentrancy.
<RESULT_START>{{"label": 1, "reason": "Vulnerable: Critical `Transfer` event is emitted *after* an external call. An attacker can re-enter before this event, leading to out-of-order event logging which can break reliant systems."}}<RESULT_END>
```

---

### Solidity Function or Modifier to Analyze

```solidity"""
                prompt2 = """
```
"""
                prompt = prompt1 + func_code + prompt2
    
            # Gọi API Gemini để phân loại
            max_retries = 6
            for attempt in range(max_retries):
                try:
                    model = genai.GenerativeModel('models/gemini-2.5-flash-preview-05-20')
                    response = model.generate_content(
                        contents=[prompt],
                        generation_config=types.GenerationConfig(
                            max_output_tokens=8192,
                            temperature=0.6
                        )
                    )
                    
                    # Trích xuất nhãn và lý do từ phản hồi
                    result_match = re.search(r'<RESULT_START>(.*?)<RESULT_END>', response.text, re.DOTALL)
                    if result_match:
                        result = json.loads(result_match.group(1))
                        label = result['label']
                        reason = result['reason']
                        
                        labels[func_name] = int(label)
                        reasons[func_name] = reason
                        api_manager.retry_count = 0  # Reset retry count on success
                        break
                    else:
                        logging.warning(f"Không tìm thấy định dạng kết quả hợp lệ cho hàm {idx + 1} trong file")
                        print(f"Không tìm thấy định dạng kết quả hợp lệ cho hàm {idx + 1} trong file")
                        raise ValueError("Định dạng kết quả không hợp lệ")
                        
                except Exception as e:
                    error_msg = str(e)
                    logging.error(f"Lỗi khi phân loại hàm {idx + 1} trong file, lần thử {attempt + 1}: {error_msg}")
                    print(f"Lỗi khi phân loại hàm {idx + 1} trong file, lần thử {attempt + 1}: {error_msg}")
                    
                    # Check for 429 error and handle key rotation
                    if "429" in error_msg and "exceeded your current quota" in error_msg and attempt % 2 == 0 and attempt > 0 and attempt < max_retries - 1:
                        api_manager.mark_current_key_error(error_msg)
                        new_key = api_manager.rotate_key()
                        if new_key:
                            logging.info(f"Rotating to new API key: {new_key[:4]}****")
                            print(f"Rotating to new API key: {new_key[:4]}****")
                            genai.configure(api_key=new_key)
                            attempt = -1  # Reset attempt to 0 after increment
                            continue
                        else:
                            logging.error("No more API keys available to rotate.")
                            print("No more API keys available to rotate.")
                            labels[func_name] = 0
                            reasons[func_name] = f"Error: No valid API keys after {max_retries} retries: {error_msg}"
                            break
                    elif attempt == max_retries - 1:
                        logging.error(f"Thất bại sau {max_retries} lần thử cho hàm {idx + 1} trong file")
                        print(f"Thất bại sau {max_retries} lần thử cho hàm {idx + 1} trong file")
                        labels[func_name] = 0
                        reasons[func_name] = f"Lỗi phân loại: {error_msg}"
                    time.sleep(2 ** attempt)  # Backoff theo cấp số nhân
    
        return labels, reasons

In [9]:
import re
import pandas as pd

def VFunction_Classification(functions_with_syntax, vulnerability, isVul):
    # list_func: array list func
    # vulnerability: "reentrancy", "timestamp"
    # isVul : 1, 0
        
    vulnerable_label_dict, vulnerable_reason_dict = classify_functions_vulnerability(vulnerability.lower(), functions_with_syntax)
    if len(vulnerable_label_dict) != len(functions_with_syntax):
        logging.error(f"Mismatch in label count: {len(vulnerable_label_dict)} labels for {len(functions_with_syntax)} functions")
        return none

    if len(vulnerable_reason_dict) != len(functions_with_syntax):
        logging.error(f"Mismatch in reason count: {len(vulnerable_reason_dict)} reasons for {len(functions_with_syntax)} functions")
        print(f"Mismatch in reason count: {len(vulnerable_reason_dict)} reasons for {len(functions_with_syntax)} functions")
        return none

    return vulnerable_label_dict, vulnerable_reason_dict

## Call Graph Only

In [10]:
import re
import os
import json
import logging
from slither import Slither
import networkx as nx

def clean_name_function(func_name):
    return func_name.replace("_", ".")

def get_call_graph(contract_path, vulnerability_type, isVul):
    """Generate call graph and return reasons as {index: reason} dictionary."""
    # Get Solidity version
    sc_version = '0.4.24'
    pattern = re.compile(r'\d\.\d\.\d+')
    try:
        with open(contract_path, 'r') as f:
            for line in f:
                if 'pragma solidity' in line and pattern.findall(line):
                    sc_version = pattern.findall(line)[0]
                    break
    except Exception as e:
        print(f"Error reading pragma: {str(e)}")

    print(sc_version)
    solc_compiler = f'/content/ge-sc/artifacts/solc-{sc_version}'
    if not os.path.exists(solc_compiler):
        solc_compiler = f'/content/ge-sc/artifacts/solc-0.4.24'
    try:
        slither = Slither(contract_path, solc=solc_compiler)
    except Exception as e:
        print("Error compiling:", e)
        print("So change to default version 0.4.24")
        solc_compiler = f'/content/ge-sc/artifacts/solc-0.4.24'
        try:
            slither = Slither(contract_path, solc=solc_compiler)
            print("Fixed sucessfully!")
        except Exception as e2:
            print("Still error so give up!")
            return
        pass

    # Extract all functions
    all_functionss = [compilation_unit.functions for compilation_unit in slither.compilation_units]
    all_modifierss = [compilation_unit.modifiers for compilation_unit in slither.compilation_units]
    all_functions = [item for sublist in all_functionss for item in sublist]
    all_modifiers = [item for sublist in all_modifierss for item in sublist]
    all_functions = all_functions + all_modifiers
    all_functions_as_dict = {function.canonical_name: function for function in all_functions}

    # Build call graph
    file_name_sc = contract_path.split('/')[-1]
    all_contracts_call_graph = _process_functions(all_functions_as_dict.values(), file_name_sc)

    return all_contracts_call_graph

# Sol Files to Fcg Files

In [11]:
from transformers import RobertaTokenizer, RobertaModel
import torch

seed_everything(42)

# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")
model = RobertaModel.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")

def get_embeddings(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)

    # Get the model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the embeddings (we use the embeddings of the [CLS] token)
    embeddings = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

    return embeddings

def extract_function_code(source_file, start, length):
    with open(source_file, 'r') as f:
        source_code = f.read()

    # Extract the function's code
    function_code = source_code[start:start + length]

    return function_code

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [12]:
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg")

In [13]:
import json
import os
from pathlib import Path
import networkx as nx
import dgl
from transformers import RobertaTokenizer, RobertaModel
import torch
import re
from functools import partial
from multiprocessing import Pool as ThreadPool
import glob


def clean_function(function_list):
    """Clean function names by removing address prefix and replacing underscores."""
    cleaned_functions = [re.sub(r"0x[0-9a-z]+\.sol_\d+_", "", func).replace("_", ".") for func in function_list]
    return cleaned_functions

In [14]:
import logging
from pathlib import Path
import os
import json
import networkx as nx
import dgl
import time

def processSolFile(solFileSrc, fcgFileDst):
    time.sleep(30)
    """Process a Solidity file to generate a DGL graph with contract_index and per-function labels."""
    fcgFileDst = Path(fcgFileDst)
    fcg_file = fcgFileDst / f'{Path(solFileSrc).stem}.fcg'
    print(f"Processing {solFileSrc}")

    vulnerability_type = "reentrancy" if "Reentrancy" in str(solFileSrc) else "timestamp"
    syntax_patterns = ["call.value"] if "reentrancy" in vulnerability_type.lower() else ["now", "block.timestamp"]
    isVul = 0 if "NonVulnerable" in str(solFileSrc) else 1
    print(f"Type: {vulnerability_type}")
    print(f"IsVul: {isVul}")
    
    G = get_call_graph(solFileSrc, vulnerability_type, isVul)
    G = nx.DiGraph(G)
    
    if len(G.nodes()) == 0:
        print(f"Compiler failed on: {solFileSrc}")
        return None
    
    mappings, mappingsH, functions_with_syntax, contract_indices, mapping_labels, mapping_code, syntax_reasons_dict, node_mapping = {}, {}, {}, {}, {}, {}, {}, {}
    katz = nx.katz_centrality(G)
    closeness = nx.closeness_centrality(G)
    clustering = nx.clustering(G)
        
    # Initialize reasons dictionary and labels 
    function_labels = [0] * len(G.nodes())
    contract_names = [data['contract_name'] for node, data in G.nodes(data=True)]
    unique_contracts = sorted(set(contract_names))
    contract_to_idx = {name: idx for idx, name in enumerate(unique_contracts)}
    
    for idx, (node, data) in enumerate(G.nodes(data=True)):
        mappings[node] = [
            G.in_degree(node),
            G.out_degree(node),
            katz[node],
            closeness[node],
            clustering[node]
        ]
        code_start = data['node_source_code_start']
        code_length = data['node_source_code_length']
        function_name = data['label'].split(".sol_")[1]
        function_name = function_name.replace("_", ".") 
        function_code = extract_function_code(solFileSrc, code_start, code_length)
        for syntax_pattern in syntax_patterns:
            if syntax_pattern in function_code:
                functions_with_syntax[function_name] = function_code
                break
        mapping_code[function_name] = function_code
        node_mapping[function_name] = idx
        syntax_reasons_dict[function_name] = "No reason provided"
        mappingsH[node] = get_embeddings(extract_function_code(solFileSrc, code_start, code_length))
        contract_indices[node] = contract_to_idx[data['contract_name']]

    # Classify functions with syntax
    if len(functions_with_syntax) > 0:
        syntax_labels, syntax_reasons = VFunction_Classification(functions_with_syntax, vulnerability_type, isVul)
        function_names = [k for k, v in functions_with_syntax.items()]
        function_codes = [v for k, v in functions_with_syntax.items()]
        
        # Update labels and reasons
        for name, code in zip(function_names, function_codes):
            idx = node_mapping[name]
            function_labels[idx] = syntax_labels[name]
            syntax_reasons_dict[name] = syntax_reasons[name]

    for node, data in G.nodes(data=True):
        function_name = data['label'].split(".sol_")[1]
        function_name = function_name.replace("_", ".")
        idx = node_mapping[function_name]
        mapping_labels[node] = function_labels[idx]
    
    nx.set_node_attributes(G, mappings, 'features')
    nx.set_node_attributes(G, mappingsH, 'featuresH')
    nx.set_node_attributes(G, contract_indices, 'contract_index')
    nx.set_node_attributes(G, mapping_labels, 'node_label')
    
    cg = nx.convert_node_labels_to_integers(G)
    
    dg = dgl.from_networkx(cg, node_attrs=['features', 'featuresH', 'contract_index', 'node_label'])

    # Save DGL graph if it doesn't exist
    if not os.path.exists(fcg_file):
        dgl.data.utils.save_graphs(str(fcg_file), [dg])
        print(f"Saved FCG: {fcg_file}")

    print(function_labels)
    
    mapping_node_code = {
        "node": node_mapping,
        "code": mapping_code,
        "reason": syntax_reasons_dict
    }
    json_path = fcgFileDst / f'{Path(solFileSrc).stem}_mapping.json'
    os.makedirs(fcgFileDst, exist_ok=True)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(mapping_node_code, f, indent=4, ensure_ascii=False)
    print(f"Saved mapping: {json_path}")
  
    return node_mapping, mapping_code, syntax_reasons_dict

In [15]:
"""Process Solidity files in parallel and generate FCG files with mappings."""
seed_everything(42)

src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
# vtypes = ["ReentrancyDataset", "TimestampDependencyDataset"]
# common_path = ["Test/NonVulnerable", "Test/Vulnerable", "Train/NonVulnerable", "Train/Vulnerable"]
# save_common = ["Test/NonVulnerable_Fcg", "Test/Vulnerable_Fcg", "Train/NonVulnerable_Fcg", "Train/Vulnerable_Fcg"]
vtypes = ["ReentrancyDataset"]
common_path = ["Test/Vulnerable", "Train/Vulnerable"]
save_common = ["Test/Vulnerable_Fcg", "Train/Vulnerable_Fcg"]

for tp in vtypes:
    for idx in range(len(common_path)):
        source = os.path.join(src, tp, common_path[idx])
        destination = os.path.join(src2, tp, save_common[idx])
        
        print(f"Source: {source}")
        print(f"Destination: {destination}")
        print("-" * 100)
        
        sol_files = glob.glob(os.path.join(source, "*.sol"))
        fcg_files = [f[:-4] for f in os.listdir(destination) if f.endswith('.fcg')]
        sol_stems = [os.path.basename(f)[:-4] for f in sol_files]
        
        checkpoint = set(sol_stems) - set(fcg_files)
        cp_sol_files = [f for f in sol_files if os.path.basename(f)[:-4] in checkpoint]
        
        print(f"Unprocessed files: {len(cp_sol_files)}")
        
        with ThreadPool(2) as pool:
            results = pool.map(partial(processSolFile, fcgFileDst=destination), cp_sol_files)


# src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0xae5a801527695745d42034eb236662927ab1f95b.sol"
# dst = "/kaggle/working/"
# tmp = processSolFile(src, dst)

Source: /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable
Destination: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg
----------------------------------------------------------------------------------------------------
Unprocessed files: 73
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/0x36f0deb0af8ab453b6b4fcc8b0b7fe2f1b44e55f.solProcessing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/15458.sol

Type: reentrancyType: reentrancy

IsVul: 1IsVul: 1

0.4.190.4.20

Phân loại hàm 1/1 cho reentrancy trong file
Phân loại hàm 1/1 cho reentrancy trong file
Lỗi khi phân loại hàm 1 trong file, lần thử 1: Expecting property name enclosed in double quotes: line 1 colum

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg/15458.fcg
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg/15458_mapping.json


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg/0x36f0deb0af8ab453b6b4fcc8b0b7fe2f1b44e55f.fcg
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg/0x36f0deb0af8ab453b6b4fcc8b0b7fe2f1b44e55f_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/0x69beaaf17c42508f92b0d72c8085b725207d65a3.sol
Type: reentrancy
IsVul: 1
0.4.20
Error compiling: Invalid compilation: 
Invalid solc compilation /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable/0x69beaaf17c42508f92b0d72c8085b725207d65a3.sol:29:13: Error: Expected identifier, got 'LParen'
 co

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0xb73f8f75cc233ec7a451d44859e06167e47c1942.fcg
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0xb73f8f75cc233ec7a451d44859e06167e47c1942_mapping.json
Lỗi khi phân loại hàm 1 trong file, lần thử 4: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Lỗi khi phân loại hàm 1 trong file, lần thử 5: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x872098e7e008079040a03efcdf313ff1911769dc.sol
Type: reentrancy
IsVul

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x23a91059fdc9579a9fbd0edc5f2ea0bfdb70deb4.fcg
[0, 0, 0, 0, 0]
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x23a91059fdc9579a9fbd0edc5f2ea0bfdb70deb4_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x15d92e219cfe22b7515dd6d1cf5a6a65a4e2acf1.sol
Type: reentrancy
IsVul: 1
0.4.21
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x1ed8691cea15e9573282175ffa3e23281fce85c0.sol
Type: reentrancy
IsVul: 1
0.4.19
Phân loại hàm 1/1 cho reentrancy trong file
Phân loại hàm 1/1 cho reentrancy trong file
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/

In [16]:
# import dgl
# graphs, _ = dgl.data.utils.load_graphs("/path/to/your/file.fcg")
# print(graphs[0].ndata['label'])  # In ra tensor chứa nhãn [0, 1, ...]

In [17]:
# src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
# src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
# types = ["TimestampDependencyDataset"]
# common_path = ["Test/Vulnerable", "Train/Vulnerable"]
# save_common = ["Test/Vulnerable_Fcg", "Train/Vulnerable_Fcg"]

# for tp in types:
#     for idx in range(len(common_path)):
#         source = os.path.join(src, tp, common_path[idx])
#         destination = os.path.join(src2, tp, save_common[idx])
        
#         print(f"Source: {source}")
#         print(f"Destination: {destination}")
#         print("-" * 100)
        
#         sol_files = glob.glob(os.path.join(source, "*.sol"))
#         fcg_files = [f[:-4] for f in os.listdir(destination) if f.endswith('.fcg')]
#         sol_stems = [os.path.basename(f)[:-4] for f in sol_files]
        
#         checkpoint = set(sol_stems) - set(fcg_files)
#         cp_sol_files = [f for f in sol_files if os.path.basename(f)[:-4] in checkpoint]
        
#         print(f"Unprocessed files: {len(cp_sol_files)}")
        
#         with ThreadPool(2) as pool:
#             results = pool.map(partial(processSolFile, fcgFileDst=destination), cp_sol_files)